In [1]:
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

In [91]:
train = pd.read_csv("data/baseline_modeling/baseline_train_model_ready.csv")
val = pd.read_csv("data/baseline_modeling/baseline_val_model_ready.csv")
test = pd.read_csv("data/baseline_modeling/baseline_test_model_ready.csv")


In [93]:
train.info()


<class 'pandas.DataFrame'>
RangeIndex: 1896 entries, 0 to 1895
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   checkpoint                1896 non-null   str    
 1   position                  1896 non-null   str    
 2   is_home                   1896 non-null   bool   
 3   formation                 1896 non-null   str    
 4   subbed                    1896 non-null   bool   
 5   last15_sprints            1896 non-null   int64  
 6   last15_hsr                1896 non-null   int64  
 7   last15_distance           1896 non-null   float64
 8   last15_mean_max_speed     1896 non-null   float64
 9   last15_peak_speed         1896 non-null   float64
 10  last15_shots              1896 non-null   int64  
 11  last15_shots_on_target    1896 non-null   int64  
 12  last15_shots_under_press  1896 non-null   int64  
 13  last15_shots_top_third    1896 non-null   int64  
 14  cumul_sprints      

In [95]:
target = "scored_after"

X_vars = train.drop(columns=target).columns.to_list()

cat_cols = train[X_vars].select_dtypes(include=["object", "category", "string"]).columns.to_list()
num_cols = train[X_vars].select_dtypes(include=["number", "bool"]).columns.to_list()

In [97]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)

X_train_np = preprocessor.fit_transform(train[X_vars])
X_val_np = preprocessor.transform(val[X_vars])

y_train_np = train[target].to_numpy()
y_val_np = val[target].to_numpy()

In [99]:
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).view(-1, 1)

X_val = torch.tensor(X_val_np, dtype=torch.float32)
y_val = torch.tensor(y_val_np, dtype=torch.float32).view(-1, 1)

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

In [101]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [105]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [107]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_y = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu())
            all_y.append(y_batch.cpu())

    all_probs = torch.cat(all_probs).numpy().ravel()
    all_y = torch.cat(all_y).numpy().ravel()

    avg_loss = total_loss / len(data_loader.dataset)
    auc = roc_auc_score(all_y, all_probs)
    preds = (all_probs >= 0.5).astype(int)
    acc = (preds == all_y).mean()

    return avg_loss, acc, auc

In [109]:
n_epochs = 100

for epoch in range(n_epochs):
    model.train()
    train_loss_sum = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * X_batch.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)
    val_loss, val_acc, val_auc = evaluate_model(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch+1:02d}/{n_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val ROC AUC: {val_auc:.4f}"
    )

Epoch 01/100 | Train Loss: 0.3934 | Val Loss: 0.2620 | Val Acc: 0.9328 | Val ROC AUC: 0.5193
Epoch 02/100 | Train Loss: 0.2112 | Val Loss: 0.2476 | Val Acc: 0.9328 | Val ROC AUC: 0.6211
Epoch 03/100 | Train Loss: 0.2026 | Val Loss: 0.2375 | Val Acc: 0.9328 | Val ROC AUC: 0.6601
Epoch 04/100 | Train Loss: 0.1991 | Val Loss: 0.2378 | Val Acc: 0.9328 | Val ROC AUC: 0.6574
Epoch 05/100 | Train Loss: 0.1972 | Val Loss: 0.2378 | Val Acc: 0.9328 | Val ROC AUC: 0.6616
Epoch 06/100 | Train Loss: 0.1906 | Val Loss: 0.2393 | Val Acc: 0.9328 | Val ROC AUC: 0.6576
Epoch 07/100 | Train Loss: 0.1879 | Val Loss: 0.2423 | Val Acc: 0.9328 | Val ROC AUC: 0.6514
Epoch 08/100 | Train Loss: 0.1861 | Val Loss: 0.2405 | Val Acc: 0.9328 | Val ROC AUC: 0.6559
Epoch 09/100 | Train Loss: 0.1868 | Val Loss: 0.2442 | Val Acc: 0.9328 | Val ROC AUC: 0.6496
Epoch 10/100 | Train Loss: 0.1807 | Val Loss: 0.2476 | Val Acc: 0.9328 | Val ROC AUC: 0.6431
Epoch 11/100 | Train Loss: 0.1796 | Val Loss: 0.2454 | Val Acc: 0.9328

In [111]:
X_test_np = preprocessor.transform(test[X_vars])
y_test_np = test[target].to_numpy()

X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32).view(-1, 1)

test_ds = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

test_loss, test_acc, test_auc = evaluate_model(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test ROC AUC: {test_auc:.4f}")

Test Loss: 0.5753 | Test Acc: 0.9210 | Test ROC AUC: 0.5851
